### Block 0 — Install packages

In [ ]:
!pip install -q openai datasets scikit-learn tqdm pandas


### Block 1 — Imports and Gemini API client

In [ ]:
import os
import getpass
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from openai import OpenAI


os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google API Key here: ")


client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],

    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


MODEL_NAME = "gemini-2.5-flash"

print(f"Client configured for model: {MODEL_NAME}")

Paste your Google API Key here: ··········
Client configured for model: gemini-2.5-flash


### Block 2 — Load FPB dataset

In [ ]:
dataset = load_dataset("ChanceFocus/en-fpb", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3 — Labels and normalization

In [ ]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map the raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Handle short variants like "pos" or "neg"
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4 — Gemini sentiment classifier

In [ ]:
def build_prompt(text: str) -> str:
    """
    Build the instruction prompt for sentiment classification.
    """
    return (
        "You are an expert financial sentiment classifier.\n\n"
        "Classify the sentiment of the following financial news snippet as exactly one of:\n"
        "negative, neutral, positive.\n\n"
        "Return only one word: negative, neutral, or positive.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )


def predict_label(text: str) -> str:
    """
    Call DeepSeek (official API) and return a normalized label.
    """
    prompt = build_prompt(text)

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a financial sentiment classifier. "
                    "Always reply with exactly one word: negative, neutral, or positive."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_tokens=8,
        temperature=0.0,
    )

    raw = completion.choices[0].message.content
    return normalize_prediction(raw)


### Block 5 — Evaluation loop over FPB test split

In [ ]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example["text"]
    true_label = example["answer"].strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 970/970 [06:32<00:00,  2.47it/s]


### Block 6 — Compute metrics and inspect predictions

In [ ]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.6010

Classification report:
              precision    recall  f1-score   support

    negative     1.0000    0.0086    0.0171       116
     neutral     0.5985    1.0000    0.7489       577
    positive     1.0000    0.0181    0.0355       277

    accuracy                         0.6010       970
   macro avg     0.8662    0.3422    0.2671       970
weighted avg     0.7612    0.6010    0.4576       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from L